# Tutorial 06 · Uncertainty sampling with the GP

In this tutorial you build the sampler of Session 6 yourself: **GP uncertainty sampling on choice and response time**.

Ready-made: a **simulated player** that plays the trials you give it, and the **GP** of Session 6 that fits all data so far. You write the part that **chooses the next trials**.

1. Start: 50 random trials (ready)
2. Iteration 1, step by step: plausible answers, expected gain from choice and RT, choose (you)
3. Iteration 2: your sampler as one function (you)
4. All remaining iterations up to 500 trials (ready)
5. Evaluation: your sampler against random (ready, as in the project)

Everything is simulated: one model (a diffusion model) answers every trial.

## Setup

In [ ]:
import os
import sys

# the course folder is one level above tutorials/ (two levels above tutorials/solutions/)
root = os.path.abspath("..")
if not os.path.isdir(os.path.join(root, "resources")):
    root = os.path.abspath(os.path.join("..", ".."))

sys.path.insert(0, root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from resources.gp import GP
from resources.twoafc_tools import (SimulatedPlayer, random_trials, candidate_grid, play_block,
                                    cross_validate, plot_iteration, plot_curves)

rng = np.random.default_rng(1)

## 1. Start: 50 random trials

`player.run(trials)` plays a table of trials (columns `ratio`, `scatteredness`) and returns one row per trial:

- `choice_blue`: 1 = blue, 0 = orange, −1 = no answer in time
- `rt_ms`: the response time

`random_trials(n, rng)` draws n trials from the random sampler (ratio between 0.1 and 0.9, scatteredness between 0 and 1). We start with 50 of them.

In [ ]:
player = SimulatedPlayer(seed=1)
data = player.run(random_trials(50, rng))
data.head()

## 2. Iteration 1, step by step

### The GP (ready)
The model is the Gaussian process of Session 6: one GP for the choice, one for the log response time, both over ratio × scatteredness. It is fitted to all trials so far.

In [ ]:
gp = GP().fit(data)
fig = plot_curves(gp)

### The candidates (ready)
Your sampler chooses from a grid of possible trials: 21 ratios × 6 scatteredness values = 126 candidates.

In [ ]:
candidates = candidate_grid()
len(candidates)

### Exercise 2.1: Plausible answers for the choice
For the choice, the GP gives at every candidate a **latent value** f with a mean and a variance: `mean, var = gp.latent(candidates)`. The probability of a blue answer is P(blue) = 1 / (1 + e^(−f)).

The variance is what the GP does not know yet. Draw **200 plausible latent functions**: f = mean + √var · z with z from a standard normal distribution (`rng.standard_normal`), one row per draw. Turn them into P(blue).

Result: `p_curves` with shape (200, number of candidates), just like the bootstrap curves of the slides.

In [ ]:
mean, var = ...       # ADD YOUR CODE HERE
f = ...               # 200 draws: shape (200, len(candidates))
p_curves = ...        # P(blue) of every draw

p_curves.shape

### Exercise 2.2: Expected gain from the choice
The split of Session 6:

- H(Y) = h(average of P(blue) over the draws): how unpredictable the answer is
- H(Y | X) = average over the draws of h(P(blue)): the noise
- expected gain I(X;Y) = H(Y) − H(Y | X)

with h(p) = −p · log₂ p − (1 − p) · log₂(1 − p). Write `h2(p)` (clip p to [1e-12, 1 − 1e-12] first, so log₂ 0 never happens) and compute `gain_choice` for every candidate.

In [ ]:
def h2(p):
    # ADD YOUR CODE HERE
    ...

gain_choice = ...   # ADD YOUR CODE HERE

gain_choice.max()

### Exercise 2.3: Expected gain from the response time
For the RT, `gp.rt_prediction(candidates)` returns three arrays: the predicted log RT, `var_f` (how uncertain the RT **curve** is at each candidate) and `var_n` (the trial noise of a single RT).

The RT is normally distributed, so the expected gain of one RT is

  ½ · log₂(1 + var_f / var_n)   bits.

(The halving-the-SE idea of Session 4: the more uncertain the curve compared with the noise, the more one RT tells.)

In [ ]:
prediction, var_f, var_n = ...   # ADD YOUR CODE HERE
gain_rt = ...                    # ADD YOUR CODE HERE

gain_rt.max()

### Exercise 2.4: Choose the next trials
Both gains are in bits, so they can simply be added: `gain = gain_choice + gain_rt`. Choose the **10 candidates with the largest gain** (hint: `np.argsort`) and store them as `next_trials`, a table with the columns `ratio` and `scatteredness`.

In [ ]:
gain = ...          # ADD YOUR CODE HERE
next_trials = ...   # ADD YOUR CODE HERE

fig = plot_iteration(gp, candidates, gain_choice, gain_rt, next_trials, data)

Grey: the trials so far. Red: the 10 trials you chose. Where does the choice gain point, where the RT gain?

### Play them (ready)
The player plays your 10 trials; they are added to the data. That completes iteration 1.

In [ ]:
data = pd.concat([data, player.run(next_trials)], ignore_index=True)
len(data)

## 3. Exercise: Iteration 2 with one function

Put exercises 2.1 to 2.4 into one function: your **sampler**. It gets the fitted GP, the number of trials to propose and a random generator, and returns the next trials.

```
def gp_uncertainty(gp, n, rng):
    plausible answers for the choice      (2.1)
    expected gain from the choice         (2.2)
    expected gain from the RT             (2.3)
    return the n candidates with the largest total gain   (2.4)
```

The cell below it runs iteration 2 with your function.

In [ ]:
def gp_uncertainty(gp, n, rng):
    candidates = candidate_grid()
    # ADD YOUR CODE HERE
    ...

In [ ]:
gp = GP().fit(data)
next_trials = gp_uncertainty(gp, 10, rng)
data = pd.concat([data, player.run(next_trials)], ignore_index=True)
len(data)

## 4. All remaining iterations (ready)

Each iteration repeats three steps until the player has played **500 trials**:

1. fit the GP to all trials so far,
2. your sampler `gp_uncertainty` proposes the next 10 trials,
3. the player plays them, and they are added to the data.

Along the way we record the largest expected gain of each iteration, to see how much the best trial could still tell. Run the cell; it takes about a quarter of a minute.

In [ ]:
largest_gain = []
while len(data) < 500:
    gp = GP().fit(data)                                              # 1. fit the GP
    largest_gain.append(gp.information(candidate_grid(), rng=rng)["total"].max())
    next_trials = gp_uncertainty(gp, 10, rng)                        # 2. your sampler proposes 10 trials
    data = pd.concat([data, player.run(next_trials)], ignore_index=True)   # 3. play them, add them

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(70 + 10 * np.arange(len(largest_gain)), largest_gain)
axes[0].set_xlabel("trials")
axes[0].set_ylabel("largest expected gain (bits)")
axes[1].scatter(data["ratio"], data["scatteredness"], s=6, alpha=0.5)
axes[1].set_xlabel("ratio")
axes[1].set_ylabel("scatteredness")
axes[1].set_title("where the 500 trials went")
fig.tight_layout()

### Question
How does the largest gain change over the iterations? Where did the trials go compared with random sampling?

## 5. Evaluation (ready-made, as in the project)

The project's table: the **same GP** is fitted on each training block and scored on each test block, in bits per held-out trial over the baselines (choice + RT). Training blocks: 500 random trials, and your 500 GP-uncertainty trials. Test blocks: 500 new random trials, and a second block of your sampler (`play_block` repeats what you just did: 50 random trials, then cycles of 10 from your sampler).

In [ ]:
sampler = lambda data, n, rng: gp_uncertainty(GP().fit(data), n, rng)
train = {"random": player.run(random_trials(500, rng)), "GP-uncertainty": data}
test = {"random": player.run(random_trials(500, rng)),
        "GP-uncertainty": play_block(sampler, player, n_trials=500, rng=rng)}

cross_validate(train, test)

### Question
Which training block teaches the GP more? Compare within a column only. Would you trust a difference of about 0.01 bits from one simulated run?

## Take-away

- A sampler is just a function: data so far → next trials.
- GP uncertainty sampling: plausible answers from the GP, expected gain = noise split for the choice + ½·log₂(1 + var_f/var_n) for the RT, then the largest gains.
- The gain per cycle shrinks as the GP learns.
- The same GP evaluates the result: which data teaches it more, on held-out trials.